**Platform specification**

In [1]:
!python --version
!gcc --version

Python 3.12.8
gcc (Raspbian 10.2.1-6+rpi1) 10.2.1 20210110
Copyright (C) 2020 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



**Dependencies**

In [2]:
!echo -e "crazyflie-lib-python: " $(python -m pip list | grep "cflib")
!echo -e "dotenv: " $(python -m pip list | grep "dotenv")

crazyflie-lib-python:  cflib 0.1.33
dotenv:  dotenv 0.9.9 python-dotenv 1.2.3


**1.1 Imports**

In [1]:
import cflib.drivers.crazyradio as crazyradio   # Communication medium device
import cflib.crtp                               # Communication protocol
from cflib.crazyflie import Crazyflie           # Drone class
from cflib.crazyflie.syncCrazyflie import SyncCrazyflie # Drone class for synchronized tasks
from cflib.crazyflie.log import LogConfig
from cflib.crazyflie.syncLogger import SyncLogger
from cflib.utils import uri_helper
from cflib.utils.reset_estimator import reset_estimator

from dotenv import load_dotenv
import logging
import sys
import threading
from threading import Event
import time
import ipywidgets as widgets

**1.2 Environment Variables**

In [2]:
load_dotenv()  # reads variables from a .env file and sets them in os.environ

# URI to the Crazyflie to connect to
URIs = []
for d in range(1,4):
    URIs.append( uri_helper.uri_from_env(env=f'DRONE{str(d)}_URI', default='radio://0/80/2M/E7E7E7E7E7') )

**1.3 Configuration**

In [3]:
deck_attached_event = Event()
crazyflie_ready = Event()

**1.4 Connection**

In [4]:
# CrazyRadio (Communication Medium) - Dongles currently connected to the USB Port of the computer
radio = crazyradio.Crazyradio()
print("\n|- Crazyradio Devices ---" + "-" * 10)
dongles = []
for d in crazyradio._find_devices():
    print(f"|\t- {d.manufacturer}, {d.product} (Serial number: {d.serial_number})")
    dongles.append(d)
print("|------------------------" + "-" * 10)

# Communication protocol driver - Initialize the low-level drivers
cflib.crtp.init_drivers(enable_serial_driver=False)

interfaces = cflib.crtp.get_interfaces_status()
print(f"|- Radio Interface: {interfaces['radio']}")

# CrazyRadio - Interface scanning
print(f"|- Wait for Crazyflie scanning...", end=' ')
drones = dict() # Dictionary: { 'drone_uri': driverClass_uri}
for uri in URIs:
    drone_addr = uri.split('/')[-1]
    available = cflib.crtp.scan_interfaces(address=int(drone_addr, 16))
    if available:
        #drones.add(available[0][0])
        """
        drones[available[0][0]] = cflib.crtp.get_link_driver(
            uri=available[0][0], 
            #radio_link_statistics_callback=radio_stats_cb,
            link_error_callback=link_error_cb
        )
        """
        drones[available[0][0]] = 1
drones_uri = list(drones.keys())
print("Crazyflie available: ", drones_uri)

/home/universome/Desktop/crazyflie-unime-edge-1/.venv_cflib_edge_1/lib/python3.12/site-packages/libusb_package/__init__.py:51: DeprecationWarning: is_resource is deprecated. Use files() instead. Refer to https://importlib-resources.readthedocs.io/en/latest/using.html#migrating-from-legacy for migration advice.
  if resources.is_resource(__name__, _LIBRARY_NAME):
/home/universome/Desktop/crazyflie-unime-edge-1/.venv_cflib_edge_1/lib/python3.12/site-packages/libusb_package/__init__.py:52: DeprecationWarning: path is deprecated. Use files() instead. Refer to https://importlib-resources.readthedocs.io/en/latest/using.html#migrating-from-legacy for migration advice.
  path_resource = resources.path(__name__, _LIBRARY_NAME)



|- Crazyradio Devices -------------
|	- Bitcraze AB, Crazyradio 2.0 (Serial number: 57D5141CC2A4510C)
|----------------------------------
|- Radio Interface: Crazyradio version 5.4
|- Wait for Crazyflie scanning... Crazyflie available:  []


**1.5 Crazyflie Synchronization**

In [7]:
drone_uri = drones_uri[0]
print("|- Link", drones[drone_uri])

cf_stats = Crazyflie(rw_cache='./cache')

if not drone_uri:
    print("No drone_uri")
    sys.exit()
    
print(f"|- Wait for Crazyflie Synchronization...", end=' ')
scf = SyncCrazyflie(drone_uri, cf=cf_stats)
scf.open_link()
print("Link opened")

time.sleep(3)  # give some time to establish connection; 3 sec seems to work
    
print("|- Crazyflie - Wait for params..")
scf.wait_for_params()

|- Link 1
|- Wait for Crazyflie Synchronization... Link opened
|- Crazyflie - Wait for params..


**Pre-Checks**

In [8]:
# ---[ Pre configuration ]---
# Pre checks on Crazyflie deck, sensors..
global logconf

def information_callback(text: str):
    print(f"|- [Platform]: {text}", end='')
scf.cf.platform.fetch_platform_informations(information_callback)
print("CRTP Protocol Version: ", scf.cf.platform.get_protocol_version())

# Console configuration
def console_callback(text: str):
    print(f"|- [Console]: {text}", end='')
scf.cf.console.receivedChar.add_callback(console_callback)

# Flow deck checks callback
def param_deck_flow(_, value_str):
    global deck_attached_event
    """The flow deck that you are using, should be correctly attached to the crazyflie. 
    If it is not, it will try to fly anyway without a good position estimate and for sure is going to crash. 
    """
    value = int(value_str)
    if value:
        deck_attached_event.set()
        print(f"|- [Deck]: {_} is attached!: {value}")
    else:
        print(f"|- [Deck]: {_} is NOT attached!: {value}")

scf.cf.param.add_update_callback(group="deck", name="bcFlow2", cb=param_deck_flow)

# if not deck_attached_event.wait(timeout=20):
#     print('No flow deck detected!')
#     sys.exit(1)

# Sensor checks
# IMU - Inertial Measurement Unit
def param_imu_sensors(_, value_str):
    group, name = _.split(".")
    sensors = {"BMP3XX":"Barometer", "AK8963":"Magnetometer", "LPS25H":"Barometer"}
    value = int(value_str)
    if value:
        print(f"[IMU Sensors]: {sensors[name]} {name} is present: {value}")
    else:
        print(f"[IMU Sensors]: {sensors[name]} {name} is NOT present: {value}")

scf.cf.param.add_update_callback(group="imu_sensors", name="BMP3XX", cb=param_imu_sensors)
scf.cf.param.add_update_callback(group="imu_sensors", name="AK8963", cb=param_imu_sensors)
scf.cf.param.add_update_callback(group="imu_sensors", name="LPS25H", cb=param_imu_sensors)

# Stabilizer - Estimator checks
def param_estimator(_, value_str):
    group, name = _.split(".")
    estimator_type = { 0:"Auto select", 1:"Complementary", 2:"Extended Kalman", 3:"Unscented Kalman" }
    value = int(value_str)
    print(f"|- [Parameter] {group} - {name}: \"{estimator_type[value]} ({value})\" is used!")

scf.cf.param.add_update_callback(group="stabilizer", name="estimator", cb=param_estimator)

# Stabilizer - Controller checks - https://www.bitcraze.io/documentation/repository/crazyflie-firmware/master/functional-areas/sensor-to-control/controllers/
def param_controller(_, value_str):
    group, name = _.split(".")
    controller_type = {0:"Auto select", 1:"PID", 2:"Mellinger", 3:"INDI", 4:"Brescianini", 5:"Lee"}
    value = int(value_str)
    print(f"|- [Parameter] {group} - {name}: \"({value}) - {controller_type[value]}\" is used!")
    
scf.cf.param.add_update_callback(group="stabilizer", name="controller", cb=param_controller)

# State checks
scf.cf.param.set_value('supervisor.infdmp', '1') # When nonzero, dump information about the current supervisor state to the console log

time.sleep(1)

Exception while doing callback on port [13]

Traceback (most recent call last):
  File "/home/universome/Desktop/crazyflie-unime-edge-1/.venv_cflib_edge_1/lib/python3.12/site-packages/cflib/crazyflie/__init__.py", line 467, in run
    cb.callback(pk)
  File "/home/universome/Desktop/crazyflie-unime-edge-1/.venv_cflib_edge_1/lib/python3.12/site-packages/cflib/crazyflie/platformservice.py", line 177, in _platform_callback
    self._callback()
TypeError: information_callback() missing 1 required positional argument: 'text'



CRTP Protocol Version:  -1
|- [Console]: SUP: Supervisor info ---
|- [Console]: SUP: State: Ready to fly
|- [Console]: SUP: Conditions: (0x401)
|- [Console]: SUP:   armed (0x1): 1
|- [Console]: SUP:   isFlying (0x2): 0
|- [Console]: SUP:   isTumbled (0x4): 0
|- [Console]: SUP:   commanderWdtWarning (0x|- [Console]: 8): 0
|- [Console]: SUP:   commanderWdtTimeout (0x|- [Console]: 10): 0
|- [Console]: SUP:   emergencyStop (0x20): 0|- [Console]: 
|- [Console]: SUP:   isCrashed (0x40): 0
|- [Console]: SUP:   preflightTimeout (0x80)|- [Console]: : 0
|- [Console]: SUP:   landingTimeout (0x100):|- [Console]:  0
|- [Console]: SUP:   deckFault (0x200): 0
|- [Console]: SUP:   rpmAtArmingValid (0x400|- [Console]: ): 1
|- [Console]: SUP:   spinupTimeout (0x800): |- [Console]: 0
|- [Console]: SUP:   motorsNotResponding (0x|- [Console]: 1000): 0


**2.1 Estimator and Controller**

In [9]:
# Stabilizer Estimator selection : 0:"Auto select", 1:"Complementary", 2:"Extended Kalman", 3:"Unscented Kalman"
scf.cf.param.set_value('stabilizer.estimator', str(2))
# Stabilizer Controller selection : 1:"PID", 2:"Mellinger", 3:"INDI", 4:"Brescianini", 5:"Lee"
scf.cf.param.set_value('stabilizer.controller', str(1))

scf.cf.param.get_value("hlCommander.vtoff", timeout=60)

# Reset estimator
reset_estimator(scf) # resets the Kalman filter and makes the Crazyflie wait until it has an accurate position estimate
time.sleep(1)

|- [Parameter] stabilizer - estimator: "Auto select (0)" is used!
|- [Console]: ESTIMATOR: Using Complementary|- [Console]:  (1) estimator
|- [Parameter] stabilizer - controller: "(1) - PID" is used!
Waiting for estimator to find position...success!


**Mission**

In [10]:
# ---[ Mission ]---
#trajectory_id = 1
#duration = mission(scf, trajectory_id)

**Multi-Ranger Thread**

**Commander**

In [10]:
# ---[ Commander ]---
#commander(scf)
print("-"*30 + "\n COMMANDER CONTROL \n" + "-"*30)
global dev
global endpoint

relative_yaw=False

# High Level Commander
hl_commander = scf.cf.high_level_commander

# Commander
global commander
commander = scf.cf.commander
commander.set_client_xmode(enabled=True)

# Supervisor
global sup
sup = scf.cf.supervisor

# Arm the Crazyflie
#scf.cf.supervisor.send_arming_request(True)
#scf.cf.platform.send_arming_request(do_arm=True)
#time.sleep(1.0)

takeoff_yaw = 3.14 / 2 if relative_yaw else 0.0

# External Position Sender
position = scf.cf.extpos
recovery = False
print("READ BITFIELD", sup.read_bitfield())
print("READ BITFIELD", sup.read_state_list())

# Arm the Crazyflie
while not ('Can be armed' in sup.read_state_list()): # Can be armed ?
    if 'Is armed' in sup.read_state_list():
        break
    time.sleep(0.3)
    if bool('Is locked' in scf.cf.supervisor.read_state_list()):
        print("|- [Supervisor] - Crazyflie State: Is locked")
    continue

sup.send_arming_request(do_arm=True) # ARM
print("|- [Commander] - Drone ARMED!")

# Check Arm
while not ('Is armed' in sup.read_state_list()): # Is armed ?
    if bool('Is armed' in scf.cf.supervisor.read_state_list()):
        print("|- [Supervisor] - Crazyflie State: Is armed")
    time.sleep(0.3)
    print("b")
    continue

crazyflie_ready.set()
print("|- [Commander] - Drone IS ARMED!")

while not ('Can fly' in sup.read_state_list()): # Can fly ?
    time.sleep(0.3)
    print("c")
    continue

print("Can fly: ", bool('Can fly' in sup.read_state_list()))
print("|- [Commander] - Drone CAN FLY!")

# min = 10001 , mid = 30001, max = 60000 
#print(f"Is Flying: {bool(sup.decode_bitfield(4))}", end="")

------------------------------
 COMMANDER CONTROL 
------------------------------
READ BITFIELD 526
READ BITFIELD ['Is armed', 'Is auto armed', 'Can fly', 'Finished HL trajectory']
|- [Commander] - Drone ARMED!
|- [Commander] - Drone IS ARMED!
Can fly:  True
|- [Commander] - Drone CAN FLY!


**3.1 Roll, Pitch, Yaw Rate and Thrust parameters**

In [2]:
roll    = widgets.FloatSlider(value=0.0, min=-20.0, max=20.0, step=1.0, description='Roll:', disabled=False, continuous_update=True, 
                              orientation='horizontal', readout=True, readout_format='.1f',)
pitch   = widgets.FloatSlider(value=0.0, min=-20.0, max=20.0, step=1.0, description='Pitch:', disabled=False, continuous_update=True,
                              orientation='vertical', readout=True, readout_format='.1f',)
yawrate = widgets.IntSlider(value=0, min=-180, max=180, step=10, description='Yawrate:', disabled=False, continuous_update=True,
                            orientation='horizontal', readout=True, readout_format='d')
thrust  = 20.0
thrust_int = widgets.IntSlider(value=10001, min=10001, max=50001, step=100, description='Thrust:', disabled=False, continuous_update=True,
                               orientation='vertical', readout=True, readout_format='d')
stop = False
def emergency_stop():
    global stop
    stop = True
    return
    
button = widgets.Button(
    description='Emergency Stop',
    disabled=False,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Emergency Stop',
    icon='check' # (FontAwesome names without the `fa-` prefix)
)
button.on_click(emergency_stop)

**3.2 Commander Sender Thread**

In [11]:
def watchdog_sender(name, commander, sup):
    print(f"|- [Watchdog Sender] Name: {name}, Thrust: {thrust}")
    global roll
    global pitch
    global yawrate
    global thrust_int
    global stop
    
    # Send thrust=0 first so that the crazyflie-firmware's safety protection requirements are met (https://forum.bitcraze.io/viewtopic.php?t=4574)
    commander.send_setpoint(0.0, 0.0, 0, 0)
    time.sleep(0.01)
    
    # Send thrust of 30000 for 3 seconds
    #logging.info("Thread %s: start feed setpoint", name)
    print(f"|- [Watchdog Sender] ", end=' ')
    while 'Can fly' in sup.read_state_list():
        try:
            #print(f".", end=' ')
            commander.send_setpoint(roll.value, pitch.value, yawrate.value, thrust_int.value)
            #commander.send_setpoint_manual(0, 0, 0, 20.0, rate=False)
            time.sleep(0.01)
            if stop:
                raise KeyboardInterrupt
            
        except KeyboardInterrupt: # Let CTRL+C actually exit
            sup.send_emergency_stop_watchdog()
            sup.send_emergency_stop() # Emergency Stop Command
            print("|- [Supervisor] - Emergency Stop Command sent!")
            commander.send_stop_setpoint()
            print("|- [Commander] - Stop Command sent!")
            break
            
        except:
            print(f"*", end=' ')
            commander.send_setpoint(roll, pitch, yawrate, thrust_int)
            #commander.send_setpoint_manual(0, 0, 0, 20.0, rate=False)
            time.sleep(0.01)
            #commander.send_setpoint(roll, pitch, yawrate, thrust)
            pass
            
        sup.send_emergency_stop_watchdog()
    
    #logging.info("Thread %s: finishing", name)

x = threading.Thread(target=watchdog_sender, args=(1, commander, sup), daemon=True)
#logging.info("Main    : before running thread")
x.start()

|- [Watchdog Sender] Name: 1, Thrust: 20.0
|- [Watchdog Sender]  

In [3]:
widgets.HBox([thrust_int, pitch, roll, yawrate, button])

In [12]:
stop = True

|- [Supervisor] - Emergency Stop Command sent!
|- [Commander] - Stop Command sent!
|- [Console]: SUP: free falling!
|- [Console]: SUP: Locked, reboot required


In [16]:
scf.close_link()